In [ ]:
# =========================
# IMPORTS
# =========================
import pandas as pd, numpy as np, re, matplotlib.pyplot as plt
from collections import Counter
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import networkx as nx

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("your_file.csv").dropna(subset=["text"])
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# =========================
# BASIC FEATURES
# =========================
df['tweet_length'] = df['text'].str.len()
df['hashtags'] = df['text'].str.findall(r"#\w+")
df['sentiment'] = df['text'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['sentiment_label'] = df['sentiment'].apply(lambda x: 'Positive' if x>0 else ('Negative' if x<0 else 'Neutral'))

# =========================
# 1. TWITTER ANALYSIS
# =========================
df.groupby(df['date'].dt.date).size().plot(title="Tweets Over Time"); plt.show()

# =========================
# 2 & 5. TOPIC MODELING
# =========================
vectorizer = CountVectorizer(stop_words='english', max_features=500)
X = vectorizer.fit_transform(df['text'])
lda = LatentDirichletAllocation(n_components=3)
lda.fit(X)

# =========================
# 3. LOCATION ANALYSIS
# =========================
if 'location' in df.columns:
    df['location'].value_counts().head(10).plot(kind='bar', title="Top Locations"); plt.show()

# =========================
# 4. HASHTAG ANALYSIS
# =========================
tags = Counter([t for tags in df['hashtags'] for t in tags]).most_common(10)
plt.bar(*zip(*tags)); plt.xticks(rotation=45); plt.title("Top Hashtags"); plt.show()

# =========================
# 6. SENTIMENT ANALYSIS
# =========================
df['sentiment_label'].value_counts().plot(kind='bar', title="Sentiment"); plt.show()

# =========================
# 7. NEGATIVE TWEETS
# =========================
neg_df = df[df['sentiment'] < 0]
print("Negative Tweets:", len(neg_df))

# =========================
# 8. USER ENGAGEMENT
# =========================
if 'likes' in df.columns:
    df.groupby('username')['likes'].sum().sort_values(ascending=False).head(10).plot(kind='bar', title="Top Engagement"); plt.show()

# =========================
# 9. DASHBOARD (BASIC VIEW)
# =========================
print(df.describe())

# =========================
# 10. PLATFORM ANALYSIS
# =========================
if 'platform' in df.columns:
    df['platform'].value_counts().plot(kind='bar', title="Platform Comparison"); plt.show()

# =========================
# 11. BRAND ANALYSIS
# =========================
brands = ['Tesla','Apple','Samsung']
for b in brands:
    print(b, df[df['text'].str.contains(b, case=False)]['sentiment'].mean())

# =========================
# 12. CONTENT TYPE ANALYSIS
# =========================
df['type'] = df['text'].apply(lambda x: 'Short' if len(x)<100 else 'Long')
df['type'].value_counts().plot(kind='bar', title="Content Type"); plt.show()

# =========================
# 13. CREATIVE CAMPAIGN (TEXT OUTPUT)
# =========================
print("Campaign Idea: Promote AI Awareness using short viral tweets + hashtags")

# =========================
# 14. NETWORK ANALYSIS
# =========================
G = nx.Graph()
for user in df['username'].unique():
    G.add_node(user)
print("Nodes:", G.number_of_nodes())

# =========================
# 15. REVIEW ANALYSIS
# =========================
print("Avg Sentiment:", df['sentiment'].mean())

# =========================
# 16. GOOGLE TREND (SIMULATED)
# =========================
keywords = ['AI','EV','IPL']
for k in keywords:
    print(k, len(df[df['text'].str.contains(k, case=False)]))

# =========================
# 17. COMPETITOR ANALYSIS
# =========================
comp = ['Apple','Samsung']
for c in comp:
    print(c, df[df['text'].str.contains(c, case=False)]['sentiment'].mean())
